# YuE2 Studio 5.0.0 — Linux GPU Builder

**This builds the actual upstream YuE2 Studio backend and YuE2 engine, not ComfyUI.**

- Upstream: https://github.com/timoncool/YuE2-Studio
- Pinned release: **v5.0.0 (21 November 2026)**
- Kaggle settings: **Accelerator None**, Internet ON.
- Builds the upstream Rust `music-server` and the pinned `yue2.cpp` runtime with CUDA 12.9 (T4 / sm_75) and Vulkan.
- Adds a checksum-verified Cloudflare `cloudflared` binary.
- Does **not** bundle model weights. The Studio downloads and manages models through its own Models / Engines screens.

## Run

1. Run this notebook from top to bottom.
2. Save a Kaggle notebook version **with outputs**.
3. Attach that saved output as a private Kaggle Dataset/input to `YuE2_Studio_Cloudflare.ipynb`.

The build is substantial and can take a while. It fails loudly if CUDA 12.9 or the pinned Cloudflare binary cannot be verified; it will not silently produce a CPU-only substitute.

The Studio application is MIT-licensed. Individual model weights and optional engines have their own licences; check those before commercial use.

In [ ]:
import os, sys, json, shutil, hashlib, subprocess, textwrap, urllib.request
from pathlib import Path

ROOT = Path("/kaggle/working/YuE2-Studio")
BUNDLE = Path("/kaggle/working/yue2-studio-bundle")
TMP = Path("/kaggle/temp/yue2-studio-build")
BUNDLE.mkdir(parents=True, exist_ok=True)
TMP.mkdir(parents=True, exist_ok=True)

def run(cmd, **kwargs):
    print("+", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)), flush=True)
    subprocess.run(cmd, check=True, **kwargs)

# Kaggle builder: CPU is sufficient; the CUDA toolkit is used to compile the T4 runtime.
run(["bash", "-lc", "apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq build-essential pkg-config cmake ninja-build nasm autoconf automake libtool curl ca-certificates git unzip xz-utils libssl-dev libasound2-dev libvulkan-dev patchelf file ccache clang libclang-dev libgtk-3-dev libayatana-appindicator3-dev"])

# Pinned Vulkan SDK (the upstream Linux build requires glslc and current Vulkan headers).
vulkan_version = "1.4.363.0"
vulkan_root = TMP / "vulkansdk"
vulkan_root.mkdir(exist_ok=True)
vulkan_archive = TMP / f"vulkansdk-linux-x86_64-{vulkan_version}.tar.xz"
if not (vulkan_root / "x86_64/bin/glslc").exists():
    urllib.request.urlretrieve(f"https://sdk.lunarg.com/sdk/download/{vulkan_version}/linux/vulkansdk-linux-x86_64-{vulkan_version}.tar.xz", vulkan_archive)
    run(["tar", "-xJf", str(vulkan_archive), "-C", str(vulkan_root), "--strip-components=1"])

# CUDA 12.9 includes T4 (sm_75). Hide any preinstalled nvcc so upstream's build script
# does not mistake Kaggle's CUDA 12.x compiler for CUDA 13.
run(["bash", "-lc", "curl -fsSL -o /tmp/cuda-keyring.deb https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb && dpkg -i /tmp/cuda-keyring.deb >/dev/null && apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq cuda-nvcc-12-9 cuda-cudart-dev-12-9 libcublas-dev-12-9"])
cuda_root = Path("/usr/local/cuda-12.9")
if not (cuda_root / "bin/nvcc").exists():
    raise RuntimeError("CUDA 12.9 nvcc was not installed; stopping rather than producing a CPU-only T4 build.")

# Rust stable.
if not Path("/root/.cargo/bin/cargo").exists():
    run(["bash", "-lc", "curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal --default-toolchain stable"])
env = os.environ.copy()
env["PATH"] = f"/root/.cargo/bin:{vulkan_root}/x86_64/bin:/usr/local/bin:/usr/bin:/bin:/usr/sbin:/sbin"
env["VULKAN_SDK"] = str(vulkan_root / "x86_64")
env["CUDA12_ROOT"] = str(cuda_root)
env["YUE_CUDA_PROFILE"] = "universal"
env["YUE_ENGINE_BUILD_ROOT"] = str(TMP / "engine-build")
env["CARGO_TARGET_DIR"] = str(TMP / "cargo-target")
env["CARGO_BUILD_JOBS"] = "4"
env["YUE_BUILD_JOBS"] = "4"

if ROOT.exists():
    shutil.rmtree(ROOT)
run(["git", "clone", "--depth", "1", "--branch", "v5.0.0", "https://github.com/timoncool/YuE2-Studio.git", str(ROOT)], env=env)

# Build the actual Studio's headless Rust service and its pinned CUDA/Vulkan YuE2 engine.
run(["bash", "-lc", "cargo build --release -p music-server"], cwd=ROOT, env=env)
run(["bash", "-lc", "scripts/build-yue-runtime-linux.sh /kaggle/working/yue2-studio-bundle/yue2-cpp"], cwd=ROOT, env=env)

server = TMP / "cargo-target/release/music-server"
engine = BUNDLE / "yue2-cpp/yue-server"
if not server.is_file() or not engine.is_file():
    raise RuntimeError("Build output is incomplete: music-server or yue-server is missing.")
shutil.copy2(server, BUNDLE / "music-server")
shutil.copy2(ROOT / "LICENSE", BUNDLE / "LICENSE")
shutil.copy2(ROOT / "engines/yue2-cpp-source.json", BUNDLE / "yue2-cpp-source.json")

# Cloudflare Quick Tunnel binary, pinned and checksum-verified.
cf_version = "2026.10.3"
cf_sha256 = "053b80b0a23031a038fdbf3c073f59f82b8f1f152aa67971d7383ffafc0dbab0"
cf_path = BUNDLE / "cloudflared"
cf_url = f"https://github.com/cloudflare/cloudflared/releases/download/{cf_version}/cloudflared-linux-amd64"
urllib.request.urlretrieve(cf_url, cf_path)
digest = hashlib.sha256(cf_path.read_bytes()).hexdigest()
if digest != cf_sha256:
    cf_path.unlink(missing_ok=True)
    raise RuntimeError(f"cloudflared checksum mismatch: {digest}")
cf_path.chmod(0o755)

manifest = {
    "studio_version": "5.0.0",
    "upstream": "https://github.com/timoncool/YuE2-Studio",
    "tag": "v5.0.0",
    "built_for": "Linux x86_64, CUDA 12.9 + Vulkan, NVIDIA T4 and newer supported CUDA devices",
    "engine": json.loads((BUNDLE / "yue2-cpp-source.json").read_text()),
    "cloudflared_version": cf_version,
    "cloudflared_sha256": cf_sha256,
    "components": ["music-server", "yue2-cpp/yue-server", "yue2-cpp CUDA/Vulkan backend libraries", "cloudflared"]
}
(BUNDLE / "manifest.json").write_text(json.dumps(manifest, indent=2))
print("\nBUILD COMPLETE")
print(json.dumps(manifest, indent=2))
print("Bundle:", BUNDLE)
print("Next: Save Version with notebook outputs, then attach this output to the Studio runtime notebook.")
